In [0]:
%run ../00-common/01-Environment_Config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
from pyspark.sql import functions as f

# Get batch folders from Landing folder
landing_batch = sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
])

# Read tracked batches
if spark.catalog.tableExists(control_table):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.table(control_table).filter(f.col("status").isin("in_progress", "completed"))\
                                        .select("batch_id").distinct().collect()
        )
    ]
else:
    tracked_batches = []

# Identify earliest unprocessed batch
new_batches = sorted(list(set(landing_batch) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None

print(f"Landing Batches : {landing_batch}")
print(f"Tracked Batches : {tracked_batches}")
print(f"Next batch to process : {next_batch}")

if next_batch is None:
    dbutils.jobs.taskValues.set(key="p_batch_id", value="")
    dbutils.jobs.taskValues.set(key="has_batch", value="false")
else:
    dbutils.jobs.taskValues.set(key="p_batch_id", value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch", value="true")

Landing Batches : ['2025-01']
Tracked Batches : []
Next batch to process : 2025-01
